# 🐦 Crawling Data Tweet MBG — Skripsi Ilmal

---

## 📋 CARA PAKAI NOTEBOOK INI (BACA DULU!)

Notebook ini dibagi menjadi **6 bagian besar**:

| Bagian | Nama | Kapan Dijalankan |
|--------|------|------------------|
| **BAGIAN 0** | Setup & Mount Google Drive | ✅ **SETIAP KALI** buka Colab |
| **BAGIAN 1** | Install & Konfigurasi | ✅ **SETIAP KALI** buka Colab |
| **BAGIAN 2** | Crawl Keyword 1 | 🗓️ Hari ke-1 |
| **BAGIAN 3** | Crawl Keyword 2 | 🗓️ Hari ke-2 |
| **BAGIAN 4** | Crawl Keyword 3 | 🗓️ Hari ke-3 |
| **BAGIAN 5** | Crawl Keyword 4 | 🗓️ Hari ke-4 |
| **BAGIAN 6** | Gabung + Filter + Simpan Final | 🗓️ Setelah semua keyword selesai |

---

### ⚠️ ATURAN PENTING:
1. **Bagian 0 dan 1 WAJIB dijalankan setiap kali** kamu buka Colab baru
2. **Setiap hari jalankan 1 keyword saja** (Bagian 2, 3, 4, atau 5)
3. **Data otomatis tersimpan ke Google Drive** — aman meski Colab mati
4. **Kalau Colab mati di tengah jalan** → buka lagi, jalankan Bagian 0+1, lalu jalankan ulang bagian yang sama, otomatis lanjut dari bulan yang belum selesai
5. **Jangan tutup tab Colab** saat crawling berlangsung

---
# 🔧 BAGIAN 0 — Mount Google Drive
### ⚠️ WAJIB DIJALANKAN SETIAP KALI BUKA COLAB

**Fungsi:** Menghubungkan Colab ke Google Drive kamu, supaya semua file crawl tersimpan permanen dan tidak hilang saat session Colab mati.

**Cara jalankan:** Klik tombol ▶️ di sebelah kiri sel di bawah ini → Akan muncul popup izin → Klik **Connect to Google Drive** → Pilih akun Google kamu

In [ ]:
# ============================================================
# BAGIAN 0 — MOUNT GOOGLE DRIVE
# Jalankan ini SETIAP KALI buka Colab baru
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os

# Folder utama di Google Drive kamu
# Semua file crawl akan tersimpan di sini
DRIVE_FOLDER = '/content/drive/MyDrive/skripsi_mbg'
RAW_DIR      = f'{DRIVE_FOLDER}/raw_data'    # folder untuk file mentah per bulan
FINAL_DIR    = f'{DRIVE_FOLDER}/final'       # folder untuk dataset final

# Buat folder jika belum ada
os.makedirs(RAW_DIR,   exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

print('✅ Google Drive berhasil terhubung!')
print(f'📁 Folder utama  : {DRIVE_FOLDER}')
print(f'📁 Folder raw    : {RAW_DIR}')
print(f'📁 Folder final  : {FINAL_DIR}')

# Tampilkan file yang sudah ada (kalau ada dari sesi sebelumnya)
existing = os.listdir(RAW_DIR)
print(f'\n📊 File raw yang sudah ada: {len(existing)} file')
if existing:
    for f in sorted(existing)[:5]:
        print(f'   - {f}')
    if len(existing) > 5:
        print(f'   ... dan {len(existing)-5} file lainnya')

Mounted at /content/drive
✅ Google Drive berhasil terhubung!
📁 Folder utama  : /content/drive/MyDrive/skripsi_mbg
📁 Folder raw    : /content/drive/MyDrive/skripsi_mbg/raw_data
📁 Folder final  : /content/drive/MyDrive/skripsi_mbg/final

📊 File raw yang sudah ada: 120 file
   - kw10_hashtag_mbg_2025-01-06_2025-02-01.csv
   - kw10_hashtag_mbg_2025-02-01_2025-03-01.csv
   - kw10_hashtag_mbg_2025-03-01_2025-04-01.csv
   - kw10_hashtag_mbg_2025-04-01_2025-05-01.csv
   - kw10_hashtag_mbg_2025-05-01_2025-06-01.csv
   ... dan 115 file lainnya


---
# 🔧 BAGIAN 0,5 — Install Chromium Dependencies
### ⚠️  harus dijalankan setiap kali session Colab baru karena instalasi sistem tidak tersimpan di Drive..

**Fungsi:**  supaya proses crawl data dari x berjalan dengan lancar

**Cara jalankan:** Klik tombol ▶️ di sebelah kiri sel di bawah ini

In [ ]:
# ============================================================
# BAGIAN 0.5 — Install Chromium Dependencies
# Estimasi waktu: ~3-5 menit
# ============================================================

print('📦 Menginstall dependensi Chromium untuk Playwright...')
print('⏳ Ini butuh 3-5 menit, tunggu sampai selesai...\n')

# Install dependensi sistem yang dibutuhkan Chromium
!apt-get update -qq
!apt-get install -y -qq \
    libnss3 \
    libnspr4 \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libdrm2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpango-1.0-0 \
    libcairo2 \
    libxshmfence1

# Install Playwright dan download browser Chromium
!pip install playwright -q
!python -m playwright install chromium
!python -m playwright install-deps chromium

print('\n✅ Chromium siap digunakan!')
print('👉 Lanjut jalankan Bagian 1')

📦 Menginstall dependensi Chromium untuk Playwright...
⏳ Ini butuh 3-5 menit, tunggu sampai selesai...

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../0-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../1-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../2-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../3-gsettings-desktop-schemas_42.0-1ubuntu1_

---
# ⚙️ BAGIAN 1 — Install & Konfigurasi
### ⚠️ WAJIB DIJALANKAN SETIAP KALI BUKA COLAB (setelah Bagian 0)

**Fungsi:** Install library Tweet Harvest dan definisikan semua fungsi yang dibutuhkan.

**Cara dapat AUTH TOKEN:**
1. Buka **x.com** di browser Chrome/Firefox
2. Login dengan akun X kamu
3. Tekan **F12** (buka DevTools)
4. Klik tab **Application** (Chrome) atau **Storage** (Firefox)
5. Di panel kiri klik **Cookies** → klik **https://x.com**
6. Cari baris bernama **`auth_token`**
7. Copy nilai di kolom **Value** (panjangnya ~40 karakter)
8. Paste ke variabel `TWITTER_AUTH_TOKEN` di bawah

In [ ]:
# ============================================================
# BAGIAN 1 — INSTALL & KONFIGURASI (VERSI FINAL)
# Jalankan ini SETIAP KALI buka Colab baru
# ============================================================

# --- Install Tweet Harvest ---
print('📦 Menginstall Tweet Harvest...')
!npm install -g tweet-harvest 2>/dev/null
print('✅ Tweet Harvest siap!')

# --- Import library ---
import subprocess
import pandas as pd
import os
import time
import re
from datetime import datetime

# ============================================================
# ⚠️  GANTI INI DENGAN AUTH TOKEN KAMU
# ============================================================
TWITTER_AUTH_TOKEN = '407f0ae2acc5655246011d3dd54868d964820893'

# Validasi
if len(TWITTER_AUTH_TOKEN) < 10:
    print('❌ ERROR: Auth token belum diisi!')
else:
    print(f'✅ Auth token: {TWITTER_AUTH_TOKEN[:8]}... ({len(TWITTER_AUTH_TOKEN)} karakter)')

# ============================================================
# PATH — Tweet Harvest selalu simpan di /content/tweets-data/
# File di Drive disimpan terpisah setelah crawl selesai
# ============================================================
TWEETS_DATA_DIR = '/content/tweets-data'   # folder default Tweet Harvest
os.makedirs(TWEETS_DATA_DIR, exist_ok=True)

# Folder Google Drive (dari Bagian 0)
# RAW_DIR dan FINAL_DIR sudah didefinisikan di Bagian 0

# ============================================================
# Daftar 12 periode bulan (Jan 2025 - Des 2025)
# ============================================================
PERIODS = [
    ('2025-01-06', '2025-02-01'),   # Bulan 1  — Jan 2025
    ('2025-02-01', '2025-03-01'),   # Bulan 2  — Feb 2025
    ('2025-03-01', '2025-04-01'),   # Bulan 3  — Mar 2025
    ('2025-04-01', '2025-05-01'),   # Bulan 4  — Apr 2025
    ('2025-05-01', '2025-06-01'),   # Bulan 5  — Mei 2025
    ('2025-06-01', '2025-07-01'),   # Bulan 6  — Jun 2025
    ('2025-07-01', '2025-08-01'),   # Bulan 7  — Jul 2025
    ('2025-08-01', '2025-09-01'),   # Bulan 8  — Agu 2025
    ('2025-09-01', '2025-10-01'),   # Bulan 9  — Sep 2025
    ('2025-10-01', '2025-11-01'),   # Bulan 10 — Okt 2025
    ('2025-11-01', '2025-12-01'),   # Bulan 11 — Nov 2025
    ('2025-12-01', '2025-12-31'),   # Bulan 12 — Des 2025
]

# ============================================================
# FUNGSI UTAMA
# ============================================================

def crawl_satu_periode(keyword, since, until, nama_file_drive, limit=800):
    """
    Crawl tweet untuk 1 keyword + 1 periode waktu.

    Alur:
    1. Cek apakah file sudah ada di Drive → kalau ada, skip
    2. Crawl → Tweet Harvest simpan di /content/tweets-data/
    3. Copy file hasil ke Google Drive

    Return: jumlah tweet yang didapat
    """
    # --- Cek apakah sudah pernah di-crawl (ada di Drive) ---
    if os.path.exists(nama_file_drive):
        try:
            df_ada = pd.read_csv(nama_file_drive)
            print(f'      ⏭️  SKIP — sudah ada di Drive ({len(df_ada)} tweets)')
            return len(df_ada)
        except:
            pass  # file rusak, crawl ulang

    # --- Buat nama file sementara (hanya nama file, tanpa path) ---
    # Tweet Harvest akan simpan di /content/tweets-data/<nama_file_temp>
    nama_file_temp = os.path.basename(nama_file_drive)  # ambil nama file saja
    path_temp      = f'{TWEETS_DATA_DIR}/{nama_file_temp}'

    # Hapus file temp lama kalau ada (sisa crawl sebelumnya yang gagal)
    if os.path.exists(path_temp):
        os.remove(path_temp)

    # --- Buat query ---
    search_query = f'{keyword} since:{since} until:{until} lang:id'

    # --- Jalankan Tweet Harvest ---
    cmd = [
        'npx', '-y', 'tweet-harvest@2.6.1',
        '-o', nama_file_temp,   # ✅ nama file saja, Tweet Harvest urus path-nya
        '-s', search_query,
        '--tab', 'LATEST',
        '-l', str(limit),
        '--token', TWITTER_AUTH_TOKEN
    ]

    # Timeout 600 detik (10 menit) per periode — cukup untuk 800 tweet
    result = subprocess.run(cmd, timeout=600)

    # --- Cek hasil & copy ke Drive ---
    if os.path.exists(path_temp):
        try:
            df = pd.read_csv(path_temp)
            count = len(df)

            if count > 0:
                # Simpan ke Google Drive
                df.to_csv(nama_file_drive, index=False)
                print(f'      💾 Disimpan ke Drive: {count} tweets')

            # Hapus file temp
            os.remove(path_temp)
            return count
        except Exception as e:
            print(f'      ⚠️  Gagal baca file: {e}')
            return 0

    return 0


def crawl_satu_keyword(nama_keyword, keyword_query, keyword_index):
    """
    Crawl semua 12 bulan untuk satu keyword.
    Otomatis skip bulan yang sudah pernah di-crawl.
    """
    print(f'\n{"="*60}')
    print(f'🚀 MULAI CRAWL: {nama_keyword}')
    print(f'🔍 Query     : {keyword_query}')
    print(f'📅 Periode   : Jan 2025 — Des 2025 (12 bulan)')
    print(f'{"="*60}')

    prefix      = f'kw{keyword_index+1}_{re.sub(r"[^a-zA-Z0-9]", "_", nama_keyword)[:20]}'
    total       = 0
    file_list   = []
    waktu_mulai = datetime.now()

    for i, (since, until) in enumerate(PERIODS):
        nama_bulan      = datetime.strptime(since, '%Y-%m-%d').strftime('%B %Y')
        # Path final di Google Drive
        nama_file_drive = f'{RAW_DIR}/{prefix}_{since}_{until}.csv'

        print(f'\n  [{i+1:02d}/12] 📅 {nama_bulan}')
        print(f'         File Drive : {os.path.basename(nama_file_drive)}')

        count = crawl_satu_periode(keyword_query, since, until, nama_file_drive)

        if count > 0:
            print(f'         ✅ Didapat: {count} tweets')
            file_list.append(nama_file_drive)
            total += count
        else:
            print(f'         ⚠️  0 tweets — tidak ada postingan bulan ini')

        print(f'         📊 Total sejauh ini: {total:,} tweets')

        # Jeda 30 detik — wajib agar tidak kena rate limit X
        if i < len(PERIODS) - 1:
            print(f'         ⏳ Jeda 30 detik...')
            time.sleep(30)

    # Ringkasan
    durasi = datetime.now() - waktu_mulai
    print(f'\n{"-"*60}')
    print(f'✅ SELESAI: {nama_keyword}')
    print(f'📊 Total tweet : {total:,}')
    print(f'📁 File tersimpan : {len(file_list)}')
    print(f'⏱️  Durasi : {str(durasi).split(".")[0]}')
    print(f'{"-"*60}')

    return file_list, total


print('\n✅ Semua fungsi siap!')
print('👉 Lanjut ke BAGIAN 2 untuk mulai crawling.')

📦 Menginstall Tweet Harvest...

added 81 packages in 22s

12 packages are looking for funding
  run `npm fund` for details
✅ Tweet Harvest siap!
✅ Auth token: 407f0ae2... (40 karakter)

✅ Semua fungsi siap!
👉 Lanjut ke BAGIAN 2 untuk mulai crawling.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
# 📥 BAGIAN 2 — Crawl Keyword

**Estimasi waktu:** ~50–60 menit per sesi keyword

**Sebelum menjalankan:** Pastikan sudah menjalankan Bagian 0, 0,5 dan Bagian 1 di atas!

**Cara jalankan:** Klik ▶️ pada sel di bawah → tunggu sampai muncul teks `✅ SELESAI`

In [ ]:
# ============================================================
# CRAWL KEYWORD 1
# ============================================================

files_kw1, total_kw1 = crawl_satu_keyword(
    nama_keyword  = 'makan_bergizi_gratis',
    keyword_query = 'makan bergizi gratis',   # ✅ tanpa tanda kutip
    keyword_index = 0
)

print(f'\n🎉 Keyword 1 selesai! Total: {total_kw1:,} tweets')
print('👉 Lanjutkan dengan BAGIAN 3 (Keyword 2)')


🚀 MULAI CRAWL: makan_bergizi_gratis
🔍 Query     : makan bergizi gratis
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw1_makan_bergizi_gratis_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 5 tweets
         ✅ Didapat: 5 tweets
         📊 Total sejauh ini: 5 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw1_makan_bergizi_gratis_2025-02-01_2025-03-01.csv
      💾 Disimpan ke Drive: 52 tweets
         ✅ Didapat: 52 tweets
         📊 Total sejauh ini: 57 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw1_makan_bergizi_gratis_2025-03-01_2025-04-01.csv
      💾 Disimpan ke Drive: 38 tweets
         ✅ Didapat: 38 tweets
         📊 Total sejauh ini: 95 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw1_makan_bergizi_gratis_2025-04-01_2025-05-01.csv
      💾 Disimpan ke Drive: 64 tweets
         ✅ Didapat: 64 tweets
         📊 Total sejauh 

In [ ]:
# ============================================================
# CRAWL KEYWORD 2
# Estimasi waktu: ~50-60 menit
# ============================================================

files_kw2, total_kw2 = crawl_satu_keyword(
    nama_keyword  = 'program_makan_bergizi',
    keyword_query = 'program makan bergizi',
    keyword_index = 1
)

print(f'\n🎉 Keyword 2 selesai! Total: {total_kw2:,} tweets')
print('👉 Besok lanjutkan dengan BAGIAN 4 (Keyword 3)')


🚀 MULAI CRAWL: program_makan_bergizi
🔍 Query     : program makan bergizi
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw2_program_makan_bergiz_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 32 tweets
         ✅ Didapat: 32 tweets
         📊 Total sejauh ini: 32 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw2_program_makan_bergiz_2025-02-01_2025-03-01.csv
      💾 Disimpan ke Drive: 24 tweets
         ✅ Didapat: 24 tweets
         📊 Total sejauh ini: 56 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw2_program_makan_bergiz_2025-03-01_2025-04-01.csv
      💾 Disimpan ke Drive: 100 tweets
         ✅ Didapat: 100 tweets
         📊 Total sejauh ini: 156 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw2_program_makan_bergiz_2025-04-01_2025-05-01.csv
      💾 Disimpan ke Drive: 60 tweets
         ✅ Didapat: 60 tweets
         📊 Total

In [ ]:
# ============================================================
# CRAWL KEYWORD 3
# Estimasi waktu: ~50-60 menit
# ============================================================

files_kw3, total_kw3 = crawl_satu_keyword(
    nama_keyword  = 'program_mbg',
    keyword_query = 'program mbg',
    keyword_index = 2
)

print(f'\n🎉 Keyword 3 selesai! Total: {total_kw3:,} tweets')
print('👉 Besok lanjutkan dengan BAGIAN 5 (Keyword 4)')


🚀 MULAI CRAWL: program_mbg
🔍 Query     : program mbg
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw3_program_mbg_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 72 tweets
         ✅ Didapat: 72 tweets
         📊 Total sejauh ini: 72 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw3_program_mbg_2025-02-01_2025-03-01.csv
      💾 Disimpan ke Drive: 24 tweets
         ✅ Didapat: 24 tweets
         📊 Total sejauh ini: 96 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw3_program_mbg_2025-03-01_2025-04-01.csv
      💾 Disimpan ke Drive: 42 tweets
         ✅ Didapat: 42 tweets
         📊 Total sejauh ini: 138 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw3_program_mbg_2025-04-01_2025-05-01.csv
      💾 Disimpan ke Drive: 60 tweets
         ✅ Didapat: 60 tweets
         📊 Total sejauh ini: 198 tweets
         ⏳ Jeda 30 detik...

  [05

In [ ]:
# ============================================================
# CRAWL KEYWORD 4
# Estimasi waktu: ~50-60 menit
# ============================================================

# Keyword ini menangkap tweet yang mengandung "mbg" DAN salah satu
# kata konteks berikut: sekolah, siswa, gizi, stunting, prabowo,
# anggaran, bergizi, gratis, pelajar, kantin, murid
#
# Contoh tweet yang AKAN diambil :
#   "mbg di sekolah anak saya sudah berjalan dengan baik"
#   "anggaran mbg terlalu besar menurut saya"
# Contoh tweet yang TIDAK diambil:
#   "mbg itu singkatan apa sih" (tidak ada kata konteks)

files_kw4, total_kw4 = crawl_satu_keyword(
    nama_keyword  = 'mbg_konteks',
    keyword_query = 'mbg (sekolah OR siswa OR gizi OR stunting OR prabowo OR anggaran OR bergizi OR gratis OR pelajar OR kantin OR murid)',
    keyword_index = 3
)

print(f'\n🎉 Keyword 4 selesai! Total: {total_kw4:,} tweets')
print('👉 Lanjut ke BAGIAN 6 untuk menggabungkan semua data!')


🚀 MULAI CRAWL: mbg_konteks
🔍 Query     : mbg (sekolah OR siswa OR gizi OR stunting OR prabowo OR anggaran OR bergizi OR gratis OR pelajar OR kantin OR murid)
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw4_mbg_konteks_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 36 tweets
         ✅ Didapat: 36 tweets
         📊 Total sejauh ini: 36 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw4_mbg_konteks_2025-02-01_2025-03-01.csv
      💾 Disimpan ke Drive: 36 tweets
         ✅ Didapat: 36 tweets
         📊 Total sejauh ini: 72 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw4_mbg_konteks_2025-03-01_2025-04-01.csv
      💾 Disimpan ke Drive: 54 tweets
         ✅ Didapat: 54 tweets
         📊 Total sejauh ini: 126 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw4_mbg_konteks_2025-04-01_2025-05-01.csv
      💾 Disimpan ke Drive: 90 tweets

In [ ]:
# ============================================================
# CRAWL KEYWORD TAMBAHAN
# ============================================================

# Keyword 5: "mbg" saja tanpa filter konteks
# Ini yang paling potensial dapat banyak data
files_kw5, total_kw5 = crawl_satu_keyword(
    nama_keyword  = 'mbg_only',
    keyword_query = 'mbg',          # ← luas, tangkap semua tweet pakai kata mbg
    keyword_index = 4
)
print(f'Keyword 5 selesai: {total_kw5:,} tweets')


🚀 MULAI CRAWL: mbg_only
🔍 Query     : mbg
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw5_mbg_only_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 20 tweets
         ✅ Didapat: 20 tweets
         📊 Total sejauh ini: 20 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw5_mbg_only_2025-02-01_2025-03-01.csv
      💾 Disimpan ke Drive: 40 tweets
         ✅ Didapat: 40 tweets
         📊 Total sejauh ini: 60 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw5_mbg_only_2025-03-01_2025-04-01.csv
      💾 Disimpan ke Drive: 28 tweets
         ✅ Didapat: 28 tweets
         📊 Total sejauh ini: 88 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw5_mbg_only_2025-04-01_2025-05-01.csv
      💾 Disimpan ke Drive: 39 tweets
         ✅ Didapat: 39 tweets
         📊 Total sejauh ini: 127 tweets
         ⏳ Jeda 30 detik...

  [05/12] 📅 May 2025
        

In [ ]:
# Keyword 6: variasi penulisan lain yang umum di masyarakat
files_kw6, total_kw6 = crawl_satu_keyword(
    nama_keyword  = 'makan_gratis_sekolah',
    keyword_query = 'makan gratis sekolah',
    keyword_index = 5
)
print(f'Keyword 6 selesai: {total_kw6:,} tweets')


🚀 MULAI CRAWL: makan_gratis_sekolah
🔍 Query     : makan gratis sekolah
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw6_makan_gratis_sekolah_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 57 tweets
         ✅ Didapat: 57 tweets
         📊 Total sejauh ini: 57 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw6_makan_gratis_sekolah_2025-02-01_2025-03-01.csv
      💾 Disimpan ke Drive: 34 tweets
         ✅ Didapat: 34 tweets
         📊 Total sejauh ini: 91 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw6_makan_gratis_sekolah_2025-03-01_2025-04-01.csv
      💾 Disimpan ke Drive: 90 tweets
         ✅ Didapat: 90 tweets
         📊 Total sejauh ini: 181 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw6_makan_gratis_sekolah_2025-04-01_2025-05-01.csv
      💾 Disimpan ke Drive: 80 tweets
         ✅ Didapat: 80 tweets
         📊 Total sej

In [ ]:
# Keyword 7: fokus pada kontroversi/opini — biasanya tweet terbanyak
files_kw7, total_kw7 = crawl_satu_keyword(
    nama_keyword  = 'mbg_opini',
    keyword_query = 'mbg (korupsi OR gagal OR berhasil OR bagus OR jelek OR setuju OR tolak OR dukung OR kritik)',
    keyword_index = 6
)
print(f'Keyword 7 selesai: {total_kw7:,} tweets')


🚀 MULAI CRAWL: mbg_opini
🔍 Query     : mbg (korupsi OR gagal OR berhasil OR bagus OR jelek OR setuju OR tolak OR dukung OR kritik)
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw7_mbg_opini_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 54 tweets
         ✅ Didapat: 54 tweets
         📊 Total sejauh ini: 54 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw7_mbg_opini_2025-02-01_2025-03-01.csv
      💾 Disimpan ke Drive: 26 tweets
         ✅ Didapat: 26 tweets
         📊 Total sejauh ini: 80 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw7_mbg_opini_2025-03-01_2025-04-01.csv
      💾 Disimpan ke Drive: 16 tweets
         ✅ Didapat: 16 tweets
         📊 Total sejauh ini: 96 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw7_mbg_opini_2025-04-01_2025-05-01.csv
      💾 Disimpan ke Drive: 72 tweets
         ✅ Didapat: 72 tweets
     

In [ ]:
# Keyword 8 — fokus sentimen negatif/kritik (biasanya banyak)
files_kw8, total_kw8 = crawl_satu_keyword(
    nama_keyword  = 'mbg_kritik',
    keyword_query = 'makan bergizi (korupsi OR gagal OR masalah OR buruk OR jelek OR tolak OR protes OR kecewa OR sia-sia OR anggaran)',
    keyword_index = 7
)


🚀 MULAI CRAWL: mbg_kritik
🔍 Query     : makan bergizi (korupsi OR gagal OR masalah OR buruk OR jelek OR tolak OR protes OR kecewa OR sia-sia OR anggaran)
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw8_mbg_kritik_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 45 tweets
         ✅ Didapat: 45 tweets
         📊 Total sejauh ini: 45 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw8_mbg_kritik_2025-02-01_2025-03-01.csv
      💾 Disimpan ke Drive: 48 tweets
         ✅ Didapat: 48 tweets
         📊 Total sejauh ini: 93 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw8_mbg_kritik_2025-03-01_2025-04-01.csv
      💾 Disimpan ke Drive: 54 tweets
         ✅ Didapat: 54 tweets
         📊 Total sejauh ini: 147 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw8_mbg_kritik_2025-04-01_2025-05-01.csv
      💾 Disimpan ke Drive: 76 tweets
       

In [ ]:
# Keyword 9 — fokus implementasi di lapangan
files_kw9, total_kw9 = crawl_satu_keyword(
    nama_keyword  = 'mbg_implementasi',
    keyword_query = 'makan bergizi (sekolah OR SD OR SMP OR SMA OR siswa OR murid OR guru OR kantin OR catering OR distribusi)',
    keyword_index = 8
)


🚀 MULAI CRAWL: mbg_implementasi
🔍 Query     : makan bergizi (sekolah OR SD OR SMP OR SMA OR siswa OR murid OR guru OR kantin OR catering OR distribusi)
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw9_mbg_implementasi_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 90 tweets
         ✅ Didapat: 90 tweets
         📊 Total sejauh ini: 90 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw9_mbg_implementasi_2025-02-01_2025-03-01.csv
      ⏭️  SKIP — sudah ada di Drive (70 tweets)
         ✅ Didapat: 70 tweets
         📊 Total sejauh ini: 160 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw9_mbg_implementasi_2025-03-01_2025-04-01.csv
      ⏭️  SKIP — sudah ada di Drive (65 tweets)
         ✅ Didapat: 65 tweets
         📊 Total sejauh ini: 225 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw9_mbg_implementasi_2025-04-01_2025-05-01.csv

In [ ]:
# Keyword 10 — pakai tagar yang sering dipakai warganet
files_kw10, total_kw10 = crawl_satu_keyword(
    nama_keyword  = 'hashtag_mbg',
    keyword_query = '#MBG OR #MakanBergiziGratis OR #MakanGratis',
    keyword_index = 9
)


🚀 MULAI CRAWL: hashtag_mbg
🔍 Query     : #MBG OR #MakanBergiziGratis OR #MakanGratis
📅 Periode   : Jan 2025 — Des 2025 (12 bulan)

  [01/12] 📅 January 2025
         File Drive : kw10_hashtag_mbg_2025-01-06_2025-02-01.csv
      💾 Disimpan ke Drive: 85 tweets
         ✅ Didapat: 85 tweets
         📊 Total sejauh ini: 85 tweets
         ⏳ Jeda 30 detik...

  [02/12] 📅 February 2025
         File Drive : kw10_hashtag_mbg_2025-02-01_2025-03-01.csv
      💾 Disimpan ke Drive: 36 tweets
         ✅ Didapat: 36 tweets
         📊 Total sejauh ini: 121 tweets
         ⏳ Jeda 30 detik...

  [03/12] 📅 March 2025
         File Drive : kw10_hashtag_mbg_2025-03-01_2025-04-01.csv
      💾 Disimpan ke Drive: 36 tweets
         ✅ Didapat: 36 tweets
         📊 Total sejauh ini: 157 tweets
         ⏳ Jeda 30 detik...

  [04/12] 📅 April 2025
         File Drive : kw10_hashtag_mbg_2025-04-01_2025-05-01.csv
      💾 Disimpan ke Drive: 7 tweets
         ✅ Didapat: 7 tweets
         📊 Total sejauh ini: 164 tweets

In [ ]:
# Keyword 11 — sebutan alternatif yang dipakai masyarakat
files_kw11, total_kw11 = crawl_satu_keyword(
    nama_keyword  = 'makan_siang_gratis',
    keyword_query = 'makan siang gratis (sekolah OR prabowo OR program OR pemerintah OR siswa)',
    keyword_index = 10
)

---
# 🔀 BAGIAN 3 — Gabung, Filter & Simpan Dataset Final
### 🗓️ Jalankan SETELAH semua Bagian Crawl Keyword selesai

**Fungsi bagian ini:**
1. Mengumpulkan semua file CSV dari 4 keyword
2. Menggabungkan menjadi 1 dataset
3. Menghapus duplikat (tweet yang muncul di lebih dari 1 keyword)
4. Memfilter tweet yang tidak relevan dengan MBG
5. Menyimpan dataset bersih ke Google Drive

**Sebelum menjalankan:** Pastikan Bagian 0 dan 1 sudah dijalankan di sesi ini!

In [ ]:
# ============================================================
# BAGIAN 3 versi 1 — GABUNG + FILTER + SIMPAN DATASET FINAL
# Jalankan setelah semua keyword selesai
# ============================================================

import pandas as pd
import os

print('🔄 Memulai proses penggabungan dataset...')
print(f'📁 Membaca semua file dari: {RAW_DIR}\n')

# --- Step 1: Kumpulkan semua file CSV ---
semua_file = [
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.endswith('.csv')
]

print(f'📊 Total file ditemukan: {len(semua_file)}')

if len(semua_file) == 0:
    print('❌ Tidak ada file CSV ditemukan!')
    print('   Pastikan Bagian 2-5 sudah dijalankan.')
else:
    # --- Step 2: Baca dan gabungkan semua file ---
    dfs = []
    gagal = []

    for f in sorted(semua_file):
        try:
            df = pd.read_csv(f)
            dfs.append(df)
        except Exception as e:
            gagal.append(f)
            print(f'   ⚠️  Gagal baca: {os.path.basename(f)} — {e}')

    gabungan = pd.concat(dfs, ignore_index=True)
    print(f'\n[Step 1] Total raw gabungan        : {len(gabungan):>8,} tweets')

    # --- Step 3: Hapus duplikat berdasarkan tweet ID ---
    # Tweet yang sama mungkin tertangkap oleh lebih dari 1 keyword
    id_col = None
    for kemungkinan_id in ['tweet_id', 'id', 'id_str']:
        if kemungkinan_id in gabungan.columns:
            id_col = kemungkinan_id
            break

    if id_col:
        gabungan.drop_duplicates(subset=[id_col], keep='first', inplace=True)
        print(f'[Step 2] Setelah hapus duplikat    : {len(gabungan):>8,} tweets')
    else:
        # Kalau tidak ada ID, dedup berdasarkan teks tweet
        text_col_dedup = 'full_text' if 'full_text' in gabungan.columns else 'text'
        gabungan.drop_duplicates(subset=[text_col_dedup], keep='first', inplace=True)
        print(f'[Step 2] Setelah hapus duplikat (by text): {len(gabungan):>8,} tweets')

    # --- Step 4: Filter relevansi ---
    # Hanya pertahankan tweet yang mengandung setidaknya 1 term berikut
    TERM_RELEVAN = [
    # === NAMA PROGRAM (utama) ===
    'makan bergizi gratis',
    'makan bergizi',
    'bergizi gratis',
    'makan siang gratis',       # nama lama program, masih banyak dipakai
    'makan siang bergizi',      # variasi nama lama

    # === SINGKATAN & LEMBAGA ===
    'program mbg',
    'mbg',
    'badan gizi nasional',
    'bgn',                      # singkatan Badan Gizi Nasional
    'sppg',                     # Satuan Pelayanan Pemenuhan Gizi

    # === ISU SPESIFIK MBG yang viral di X ===
    'dapur mbg',                # "dapur MBG di sekolah", "dapur MBG tutup"
    'motor mbg',                # isu pengadaan motor BGN yang viral
    'anggaran mbg',             # perdebatan anggaran Rp71-420 triliun
    'keracunan mbg',            # kasus keracunan massal — sangat banyak di X
    'menu mbg',                 # "menu MBG hari ini", "menu MBG tidak layak"

    # === TAGAR YANG SERING DIPAKAI ===
    '#mbg',
    '#makanbergizigratis',
    '#makansianggratis',
    ]

    def cek_relevan(teks):
        """Kembalikan True jika tweet relevan dengan Program MBG."""
        if not isinstance(teks, str):
            return False
        teks_lower = teks.lower()
        for term in TERM_RELEVAN:
            if term in teks_lower:
                return True
        return False

    text_col = 'full_text' if 'full_text' in gabungan.columns else 'text'
    gabungan['relevan'] = gabungan[text_col].apply(cek_relevan)

    dibuang   = gabungan[~gabungan['relevan']]
    bersih    = gabungan[gabungan['relevan']].copy()
    bersih.drop(columns=['relevan'], inplace=True)

    print(f'[Step 3] Dibuang (tidak relevan)   : {len(dibuang):>8,} tweets')
    print(f'[Step 4] Dataset BERSIH FINAL      : {len(bersih):>8,} tweets')

    # --- Step 5: Simpan ---
    output_path = f'{FINAL_DIR}/dataset_mbg_bersih.csv'
    bersih.to_csv(output_path, index=False)

    print(f'\n✅ Dataset final tersimpan ke:')
    print(f'   {output_path}')

    # --- Step 6: Tampilkan ringkasan ---
    print(f'\n{"="*50}')
    print('📊 RINGKASAN DATASET FINAL')
    print(f'{"="*50}')
    print(f'Total tweet bersih : {len(bersih):,}')
    print(f'Kolom tersedia     : {list(bersih.columns)}')

    # Distribusi per bulan (jika ada kolom tanggal)
    date_col = None
    for kemungkinan_date in ['created_at', 'date', 'timestamp']:
        if kemungkinan_date in bersih.columns:
            date_col = kemungkinan_date
            break

    if date_col:
        # Menjadi ini (format Twitter/X yang standar):
        bersih[date_col] = pd.to_datetime(
            bersih[date_col],
            format='%a %b %d %H:%M:%S %z %Y',  # contoh: "Fri Jan 31 15:44:40 +0000 2025"
            errors='coerce'
        )
        bersih['bulan']  = bersih[date_col].dt.to_period('M')
        distribusi       = bersih.groupby('bulan').size().reset_index(name='jumlah_tweet')

        print(f'\n📅 Distribusi per bulan:')
        print(distribusi.to_string(index=False))
        print(f'\nTotal: {distribusi["jumlah_tweet"].sum():,} tweets')

    print(f'\n🎉 SELESAI! Dataset siap digunakan untuk preprocessing.')

🔄 Memulai proses penggabungan dataset...
📁 Membaca semua file dari: /content/drive/MyDrive/skripsi_mbg/raw_data

📊 Total file ditemukan: 120

[Step 1] Total raw gabungan        :    6,693 tweets
[Step 2] Setelah hapus duplikat    :    1,554 tweets
[Step 3] Dibuang (tidak relevan)   :      107 tweets
[Step 4] Dataset BERSIH FINAL      :    1,447 tweets

✅ Dataset final tersimpan ke:
   /content/drive/MyDrive/skripsi_mbg/final/dataset_mbg_bersih.csv

📊 RINGKASAN DATASET FINAL
Total tweet bersih : 1,447
Kolom tersedia     : ['conversation_id_str', 'created_at', 'favorite_count', 'full_text', 'id_str', 'image_url', 'in_reply_to_screen_name', 'lang', 'location', 'quote_count', 'reply_count', 'retweet_count', 'tweet_url', 'user_id_str', 'username']

📅 Distribusi per bulan:
  bulan  jumlah_tweet
2025-01           112
2025-02            94
2025-03            99
2025-04           101
2025-05           105
2025-06            86
2025-07           117
2025-08           146
2025-09           137
20

/tmp/ipykernel_820/2227518405.py:129: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bersih[date_col] = pd.to_datetime(bersih[date_col], errors='coerce')
/tmp/ipykernel_820/2227518405.py:130: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  bersih['bulan']  = bersih[date_col].dt.to_period('M')


---
# 🔍 BAGIAN 4 — Cek Progress (Opsional)

**Fungsi:** Cek berapa banyak data yang sudah terkumpul sejauh ini, tanpa harus menunggu semua keyword selesai.

Bisa dijalankan kapan saja untuk melihat status terkini.

In [ ]:
# ============================================================
# BAGIAN 7 — CEK PROGRESS (OPSIONAL)
# Jalankan kapan saja untuk melihat status terkini
# ============================================================

import os
import pandas as pd

semua_file = [
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.endswith('.csv')
]

if not semua_file:
    print('📭 Belum ada file yang terkumpul.')
else:
    print(f'📁 File terkumpul di Google Drive: {len(semua_file)} file\n')

    total_keseluruhan = 0
    per_keyword = {}

    for f in sorted(semua_file):
        nama = os.path.basename(f)
        # Ambil prefix keyword (misal: kw1, kw2, dst)
        prefix = nama[:4]  # kw1_, kw2_, dst

        try:
            df = pd.read_csv(f)
            jumlah = len(df)
        except:
            jumlah = 0

        if prefix not in per_keyword:
            per_keyword[prefix] = {'file': 0, 'tweet': 0}
        per_keyword[prefix]['file']  += 1
        per_keyword[prefix]['tweet'] += jumlah
        total_keseluruhan += jumlah

    print('📊 Status per keyword:')
    label_kw = {
        'kw1_': 'Keyword 1 — makan bergizi gratis',
        'kw2_': 'Keyword 2 — program makan bergizi',
        'kw3_': 'Keyword 3 — program mbg',
        'kw4_': 'Keyword 4 — mbg + konteks',
    }
    for prefix, data in sorted(per_keyword.items()):
        nama_kw = label_kw.get(prefix, prefix)
        status  = '✅ Selesai (12/12)' if data['file'] >= 12 else f'⏳ Progress ({data["file"]}/12 bulan)'
        print(f'  {nama_kw}')
        print(f'    {status} — {data["tweet"]:,} tweets')

    print(f'\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'📊 TOTAL SEMENTARA: {total_keseluruhan:,} tweets (raw, belum difilter)')
    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

📁 File terkumpul di Google Drive: 120 file

📊 Status per keyword:
  kw10
    ✅ Selesai (12/12) — 669 tweets
  Keyword 1 — makan bergizi gratis
    ✅ Selesai (12/12) — 518 tweets
  Keyword 2 — program makan bergizi
    ✅ Selesai (12/12) — 586 tweets
  Keyword 3 — program mbg
    ✅ Selesai (12/12) — 608 tweets
  Keyword 4 — mbg + konteks
    ✅ Selesai (12/12) — 737 tweets
  kw5_
    ✅ Selesai (12/12) — 663 tweets
  kw6_
    ✅ Selesai (12/12) — 682 tweets
  kw7_
    ✅ Selesai (12/12) — 562 tweets
  kw8_
    ✅ Selesai (12/12) — 841 tweets
  kw9_
    ✅ Selesai (12/12) — 827 tweets

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 TOTAL SEMENTARA: 6,693 tweets (raw, belum difilter)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
